In [1]:
import numpy as np
import math

In [2]:
"""
%   Created on Thu Sep 19 13:21:19 2021
%   @author: Chaiyut Charoenphon
%   Course : Geodetic Survey 2108415
%   
  Ref:
        Geoscience Australia - Python Geodesy Package
        Geodesy Module
        https://github.com/GeoscienceAustralia/GeodePy/blob/master/geodepy/geodesy.py
"""
from pygeodesy.ellipsoidalKarney import LatLon
import pygeodesy
from pyproj import Geod
import pyproj
import pygeodesy.dms as dms
from math import degrees, radians, sqrt, sin, cos, tan, asin, acos, atan, atan2
from geodepy.constants import grs80, utm, wgs84
from geodepy.convert import geo2grid, grid2geo, angular_typecheck

############################################################
def dd2dms(dd, PREC=5):
    #arcsecond is 5-digit decimal fraction
    return dms.toDMS(dd,prec=PREC)
def dms2dd(dms_str,SEP=' '):
    return dms.parseDMS(dms_str, sep=SEP)
    
#คำนวณรัศมีโลก ณ Latitude โดยประมาณ
def radius (B):
    # input:  B is Latitude in Degree unit
    # output: (Approximate) radius in m. unit (Approximate)
    B = radians(B) #converting into radians
    # WGS84
    a = 6378.137  #Radius at sea level at equator in m. unit
    b = 6356.752314245  #Radius at poles in m. unit
    c = (a**2*cos(B))**2
    d = (b**2*sin(B))**2
    e = (a*cos(B))**2
    f = (b*sin(B))**2
    R = sqrt((c+d)/(e+f))*1000.0
    return R

#แปลงค่าพิกัด xyz เป็น ละติจูด ลองจิจูด และ ความสูงเหนือทรงรี  หน่วยดีกรี
def xyz2lla(x,y,z):
    ecef = '+proj=geocent +ellps=WGS84' # Cartisian   
    lla = "epsg:4326" # WGS84 Geodetic    
    transproj = pyproj.Transformer.from_crs(ecef,lla)
    lat,lon,ell_h = transproj.transform(x,y,z,radians=False)
    return lat,lon,ell_h

#แปลงค่าพิกัด ละติจูด ลองจิจูด และ ความสูงเหนือทรงรี   เป็นพิกัด UTM Zone 47N
def lla2utm(lat,lon,ell_h,zone):
    '''
        I = lat ,long unit Degree
        I = ell_h unit metre
        O = E,N,h unit metre
    '''
    lla = "epsg:4326" # WGS84 Geodetic
    utm = "epsg:326" + zone # WGS84 UTM Zone 47N  
    transproj = pyproj.Transformer.from_crs(lla,utm,always_xy=True) # always_xy หมายถึงให้เรียง lon,lat
    E,N,h = transproj.transform(lon,lat,ell_h,radians=False)
    return E,N,h

#คำนวนสเกลแฟกเตอร์ในระบบพิกัด UTM โดยระบบพิกัดละติจูด ลองจิจูด และ โซน
def utm_scale_factor(lat,lon,zone):
    '''
        I = lat ,long unit Degree
        I = zone (UTM)
        O = scaleX, scaleY, scaleArea, mer_conv
    '''
    prj = pyproj.Proj('epsg:326' + zone)
    fc = prj.get_factors(lon,lat,radians=False)
    scaleX = fc.parallel_scale
    scaleY = fc.meridional_scale
    scaleArea = fc.areal_scale 
    mer_conv = fc.meridian_convergence
    return scaleX, scaleY, scaleArea, mer_conv

#คำนวนความสูงเหนือระดับน้ำทะเลปานกลาง อ้างอิงแบบจำลอง TGM2017 
def MSL(lat,lon,h):
    #  Download: tgm2017-1_2.pgm size 469 MB
    #  https://www.priabroy.name/archives/sdm_downloads/tgm2017-1
    geoid_model = r"C:\Lab_Geodetic_Survey\Lab4\tgm2017-1_2.pgm"
    geoid_interpolator = pygeodesy.GeoidKarney(geoid_model)
    # Get the geoid height
    pos = LatLon(lat, lon)
    N = geoid_interpolator(pos)
    return h-N

#แปลงค่าพิกัด xyz เป็นพิกัด Lambert Conformal Conic (LCC)
def xyz2LCC(x,y,z):   
    #Lambert Conformal Conic 1 standard parallel Conversion
    llc_1sp   = '+proj=lcc +lon_0=99 +lat_0=0 +x_0=500000 +y_0=0 +ellps=WGS84 +lat_1=33'
    #Lambert Conformal Conic conversion with scale factor
    llc_2sp_1 = '+proj=lcc +lon_0=99 +lat_0=0 +x_0=500000 +y_0=0 +ellps=WGS84 +k=0.9996'
    #Lambert Conformal Conic 2 standard parallel Conversion
    llc_2sp_2 = '+proj=lcc +lon_0=99 +lat_0=0 +x_0=500000 +y_0=0 +ellps=WGS84 +lat_1=9 +lat_2=12'
    
    ecef = {"proj":'geocent', "ellps":'WGS84'}
    transproj = pyproj.Transformer.from_crs(ecef,llc_1sp)
    E,N,h = transproj.transform(x,y,z)
    return (E,N,h)

def utm2lla(E,N,h,zone):   
    utm = "epsg:326" + zone # WGS84 UTM Zone 47N  
    lla = "epsg:4326" # WGS84 Geodetic 
    transproj = pyproj.Transformer.from_crs(utm,lla) # always_xy หมายถึงให้เรียง lon,lat
    lat,lon,h = transproj.transform(E,N,h,radians=False)
    return lat,lon,h
    
def utm2xyz(E,N,h,zone):
    utm = "epsg:326" + zone # WGS84 UTM Zone 47N  
    ecef = {"proj":'geocent', "ellps":'WGS84'}   
    transproj = pyproj.Transformer.from_crs(utm,ecef,always_xy=True) # always_xy หมายถึงให้เรียง lon,lat
    x,y,z = transproj.transform(E,N,h)
    return x,y,z
def lla2xyz(lat,lon,h):
    lla = "epsg:4326" # WGS84 Geodetic
    ecef = {"proj":'geocent', "ellps":'WGS84'}
    transproj = pyproj.Transformer.from_crs(lla,ecef,always_xy=True) # always_xy หมายถึงให้เรียง lon,lat
    x,y,z = transproj.transform(lon,lat,h,radians=False)
    return E,N,h    
def xyz2utm(x,y,z,zone):
    utm = "epsg:326" + zone # WGS84 UTM Zone 47N  
    ecef = {"proj":'geocent', "ellps":'WGS84'}
    transproj = pyproj.Transformer.from_crs(ecef,utm) # always_xy หมายถึงให้เรียง lon,lat
    E,N,h = transproj.transform(x,y,z)
    return E,N,h
#+++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++
#+++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++
def line_sf(zone1, east1, north1, zone2, east2, north2,
            hemisphere='south', ellipsoid=grs80, projection=utm):
    """
    Computes Line Scale Factor for a pair of Transverse Mercator Coordinates
    Ref: Deakin 2010, Traverse Computations on the Ellipsoid and on the
    Universal Transverse Mercator Projection, pp 35
    http://www.mygeodesy.id.au/documents/Trav_Comp_V2.1.pdf
    :param zone1: Station 1 Zone Number - 1 to 60
    :param east1: Station 1 Easting (m, within 3330km of Central Meridian)
    :param north1: Station 1 Northing (m, 0 to 10,000,000m)
    :param zone2: Station 2 Zone Number - 1 to 60
    :param east2: Station 2 Easting (m, within 3330km of Central Meridian)
    :param north2: Station 2 Northing (m, 0 to 10,000,000m)
    :param hemisphere: String - 'North' or 'South'(default)
    :param ellipsoid: Ellipsoid Object (default GRS80)
    :param projection: Projection Object (default Universal Transverse Mercator)
    :return: Line Scale Factor (relative to Zone 1 if different zones
    are entered)
    """
    # Re-project cross-zone coordinate to same UTM zone
    if zone1 != zone2:
        # Re-project Station 2 Coordinates to same zone as Station 1
        stn2_geo = grid2geo(zone2, east2, north2, hemisphere, ellipsoid)
        stn2_zone1 = geo2grid(stn2_geo[0], stn2_geo[1], zone1, ellipsoid)
        zone2 = stn2_zone1[1]
        east2 = stn2_zone1[2]
        north2 = stn2_zone1[3]

    # Comute easting distances from Central Meridian
    eastofcm1 = east1 - projection.falseeast
    eastofcm2 = east2 - projection.falseeast

    # Compute Mean Latitude
    lat1 = grid2geo(zone1, east1, north1, hemisphere, ellipsoid)[0]
    lat2 = grid2geo(zone2, east2, north2, hemisphere, ellipsoid)[0]
    lat_mean = (lat1 + lat2) / 2

    # Compute Line Scale Factor (Deakin 2010 Eq. 13)
    r_sq_m = (rho(lat_mean, ellipsoid) *
              nu(lat_mean, ellipsoid) *
              projection.cmscale ** 2)
    k1 = ((eastofcm1 ** 2 + eastofcm1 * eastofcm2 + eastofcm2 ** 2) /
          (6 * r_sq_m))
    k2 = ((eastofcm1 ** 2 + eastofcm1 * eastofcm2 + eastofcm2 ** 2) /
          (36 * r_sq_m))
    return projection.cmscale * (1 + k1 * (1 + k2))
    
def rho(lat, ellipsoid=grs80):
    """
    symbol = rho
    Return the radius of curvature of the ellipsoid in the meridian plane
    (rho) at a given latitude
    :param lat: latitude in decimal degrees
    :param ellipsoid: Ellipsoid Object
    :return: rho at specified latitude
    """
    return ((ellipsoid.semimaj * (1 - ellipsoid.ecc1sq)) /
            (1 - ellipsoid.ecc1sq * (sin(radians(lat)) ** 2)) ** 1.5)

def nu(lat, ellipsoid=grs80):
    """
    symbol = v
    Return the radius of curvature of the ellipsoid in the prime vertical plane
    (nu) at a given latitude
    :param lat: latitude in decimal degrees
    :param ellipsoid: Ellipsoid Object
    :return: nu at specified latitude
    """
    return (ellipsoid.semimaj / sqrt(1 - ellipsoid.ecc1sq * (sin(radians(lat)) ** 2)))   

def RadiusOfNormSection(lat, az12, ellipsoid=grs80):
    """
    Return the radius of  point 1 of the normal section in the direction Az12
    :param lat: latitude in decimal degrees
    :param az12: Azimuth Point 1 to Point 2 in decimal degrees
    :param ellipsoid: Ellipsoid Object default grs80
    :return: radius at specified latitude in m unit
    nu = v
    """  
    _rho = rho(lat,ellipsoid)
    _nu  = nu(lat,ellipsoid)
    return (_rho*_nu) / (_rho*(sin(radians(az12))**2) + _nu*(cos(radians(az12)))**2)   
   
def Ground_to_Ell(R,D,h1,h2):
    '''
        Convert Ground distance to ellipsoidal distance in m unit
        input:
            D is the ground distance in m. unit
            h1 and h2 are the ellipsoidal height (m. unit) at P1 and P2, respectively.
        output:
            ellipsoidal distance m. unit
    '''
    dh = h2-h1 # approximate
    c = R*sqrt((D**2-dh**2)/((R+h1)*(R+h2)))
    #c = R*D/(R+(h1+h2)*0.5)
    #print(c)
    s = 2.0*R*asin(c/(2.0*R))
    return s


#$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$


### Step 1

In [3]:
geom = Geod(ellps='WGS84')
lat2 = dms2dd("13 31 32.84002")
lon2 = dms2dd("99 44 39.81003")

lat1 = dms2dd("13 32 10.08001")
lon1 = dms2dd("99 44 22.84001") 

az12,az21,D = geom.inv(lon1,lat1,lon2,lat2) # azimuth and distance
print(f"{dd2dms(az12)} {dd2dms(az21+360)} {D:.4f} m")

print(D)
print(az21+360)

155°58′08.07208″ 335°58′12.04257″ 1253.0630 m
1253.0630076842374
335.9700118247249


In [4]:
geom = Geod(ellps='WGS84')
lat6 = dms2dd("13 32 03.97585")
lon6 = dms2dd("99 49 01.99500") 

lat5 = dms2dd("13 32 33.03943")
lon5 = dms2dd("99 48 14.09309") 

az56,az65,D = geom.inv(lon5,lat5,lon6,lat6) # azimuth and distance
print(f"{dd2dms(az56)} {dd2dms(az65+360)} {D:.4f} m")

print(D)
print(az56)

121°48′08.65294″ 301°48′19.86669″ 1694.7949 m
1694.7948644147107
121.8024035933267


### Step 2

In [5]:
sta2 = dms2dd("100 23 57.85")
sta3 = dms2dd("210 05 34.12")
sta4 = dms2dd("99 45 24.31")
sta5 = dms2dd("275 34 11.88")
print(sta2, sta3, sta4, sta5)

100.39940277777778 210.0928111111111 99.75675277777778 275.5699666666667


In [6]:
R = RadiusOfNormSection(dms2dd('13 31 32.84002'),335.97001 + 100.39940 - 360) # unit m. # จุดเริ่มต้น # az21 + hoor ang 123 = az23
D = 3234.653 # unit m. ระยะเอียง (Slope distance)
h1 = 70.345 # unit m. #Ellipsoidal height at P1
h2 = 110.243 # unit m. #Ellipsoidal height at P2
ell_distance = Ground_to_Ell(R,D,h1,h2)
ell_distance

3234.361167306442

In [7]:
az23 = 335.97001 + 100.39940 - 360
az32 = az23 + 180
az32

256.36941

In [8]:
R = RadiusOfNormSection(dms2dd('13 31 32.84002'),az32 + 210.0928111111111 - 360) # unit m. # จุดเริ่มต้น # az21 + hoor ang 123 = az23
D = 2517.356 # unit m. ระยะเอียง (Slope distance)
h1 = 110.243 # unit m. #Ellipsoidal height at P1
h2 = 20.565 # unit m. #Ellipsoidal height at P2
ell_distance = Ground_to_Ell(R,D,h1,h2)
ell_distance

2515.7323641000485

In [9]:
az34 = az32 + 210.0928111111111 - 360
az43 = az34 + 180
az43

286.46222111111115

In [10]:
R = RadiusOfNormSection(dms2dd('13 31 32.84002'),az43 + 99.75675277777778 - 360) # unit m. # จุดเริ่มต้น # az21 + hoor ang 123 = az23
D = 2009.683 # unit m. ระยะเอียง (Slope distance)
h1 = 20.565 # unit m. #Ellipsoidal height at P1
h2 = 100.021 # unit m. #Ellipsoidal height at P2
ell_distance = Ground_to_Ell(R,D,h1,h2)
ell_distance

2008.0926082465544

### Step 3

#### หา Geodetic Az

In [11]:
az21 = 335.9700118247249
az23 = az21 + 100.39940277777778 - 360
az23

76.36941460250267

In [12]:
az32 = az23 + 180
az32

256.3694146025027

In [13]:
az34 = az32 + 210.0928111111111 - 360
az34

106.46222571361375

In [14]:
az43 = az34 + 180
az43

286.46222571361375

In [15]:
az45 = az43 + 99.75675277777778 - 360
az45

26.21897849139151

In [16]:
az54 = az45 + 180
az54

206.2189784913915

In [17]:
az56 = az54 + 275.5699666666667 - 360
az56

121.7889451580582

In [18]:
az56_ref = 121.8024035933267
diff_az56_misclosure = az56_ref - az56
diff_az56_misclosure

0.013458435268503877

##### หา Lat/Lon

In [19]:
geom = Geod(ellps="WGS84")
lat = dms2dd('13 31 32.84002')
lon = dms2dd('99 44 39.81003')
az23 = dms2dd("76 22 9.89") # อย่าใช้ฟังก์ชันแปลงเด็จขาด เอา dd มากดเครื่องคิดเลข (อย่าเอา dd มาทำ dms)
gs23 = 3234.361
endlon, endlat, backaz = geom.fwd(lon, lat, az23, gs23)
print(f"{dd2dms(endlat)} {dd2dms(endlon)} {dd2dms(backaz+360)}")

13°31′57.6359″ 99°46′24.34483″ 256°22′34.34502″


In [27]:
geom = Geod(ellps="WGS84")
lat = dms2dd('13 31 57.6359')
lon = dms2dd('99 46 24.34483')
az34 = dms2dd("106 27 44.01")
gs34 = 2515.732
endlon, endlat, backaz = geom.fwd(lon, lat, az34, gs34)
print(f"{dd2dms(endlat)} {dd2dms(endlon)} {dd2dms(backaz+360)}")

13°31′34.43446″ 99°47′44.57802″ 286°28′02.78017″


In [28]:
geom = Geod(ellps="WGS84")
lat = dms2dd('13 31 34.43446')
lon = dms2dd('99 47 44.57802')
az45 = dms2dd("26 13 8.32")
gs45 = 2008.093
endlon, endlat, backaz = geom.fwd(lon, lat, az45, gs45)
print(f"{dd2dms(endlat)} {dd2dms(endlon)} {dd2dms(backaz+360)}")

13°32′33.0532″ 99°48′14.084″ 206°13′15.22525″


#### หา E/N

In [22]:
def lla2utm(lat,lon,ell_h,zone):
    lla = "epsg:4326"
    utm = "epsg:326" + str(zone)
    transproj = pyproj.Transformer.from_crs(lla, utm, always_xy=True)
    E, N, h = transproj.transform(lon, lat, ell_h, radians = False)
    return E, N, h

In [31]:
lla_sta3 = lla2utm(dms2dd('13 31 57.6359'),dms2dd('99 46 24.34483'),110.243,'47')
lla_sta3

(583691.5075942575, 1496175.832281871, 110.243)

In [32]:
lla_sta4 = lla2utm(dms2dd('13 31 34.43446'),dms2dd('99 47 44.57802'),20.565,'47')
lla_sta4

(586105.6004366875, 1495470.7584522562, 20.565)

In [33]:
lla_sta5 = lla2utm(dms2dd('13 32 33.0532'),dms2dd('99 48 14.084'),100.021,'47')
lla_sta5

(586986.6509594398, 1497274.5626383838, 100.021)

In [34]:
diffE = 586986.926 - lla_sta5[0]
diffN = 1497274.141 - lla_sta5[1]
print(f"diffE = {diffE:.3f} m, diffN = {diffN:.3f} m")

diffE = 0.275 m, diffN = -0.422 m


In [36]:
misclosure = sqrt(diffE**2 + diffN**2)
misclosure

0.5034145770638323

In [37]:
sum_dist = 3234.361167306442 + 2515.7323641000485 + 2008.0926082465544
sum_dist

7758.186139653044

In [39]:
ratio_denominator = sum_dist / misclosure
ratio_denominator

15411.127315586882